# Step 9: do more inputs help? (added after looking)

Plan and expectations committed before these results existed: see [PLAN_more_inputs.md](PLAN_more_inputs.md).

- **Sets:** A = Kim's 12 indicators (step 4), B = about 30 standard extras, C = tsfresh (picked on each fold's training years), D = A plus the last move of the S&P 500, VIX, the currency and Brent oil.
- **Same as step 4:** six markets, seven models, train up to Y−2, tune on Y−1, test on Y. Test years **2017–2026**: the spent holdout counts as development data now.
- **Same days for every set.**

`more_inputs.py` runs the models and saves them. This notebook only scores them.

In [1]:
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.metrics import roc_auc_score

from classifiers import overview
from data import load_all
from evaluation import bootstrap_range, holm, mcnemar
from models import pooled
from more_inputs import MODELS, RUNS, SAVED, run_all

markets = load_all(include_holdout=True)
preds = run_all(markets)  # loads the saved runs

def together(target, s, model):
    """One model's predictions on every market, one row per market-day."""
    return pd.concat([preds[target, s, model, m].assign(market=m) for m in markets])

def right(p):
    return (p["p_up"] > 0.5) == p["up"]

pd.Series({m: len(preds["close to close", "A", "LDA", m]) for m in markets}, name="test days").to_frame().T

,Indonesia,Malaysia,Philippines,Singapore,Thailand,India
test days,2341,2387,2386,2463,2381,2412


## Overview

Mean AUC over the 60 folds (6 markets × 10 test years), per model and set. A coin flip scores 0.50.

In [2]:
pd.DataFrame({f"{s}, {target}": overview({model: {m: preds[target, s, model, m] for m in markets} for model in MODELS})["mean AUC"]
              for target, sets in RUNS.items() for s in sets}).round(3)

,"A, close to close","B, close to close","C, close to close","D, close to close","A, open to close","D, open to close"
LDA,0.509,0.509,0.505,0.618,0.511,0.534
Logistic regression,0.516,0.514,0.506,0.623,0.513,0.536
KNN,0.512,0.515,0.507,0.607,0.508,0.527
SVM,0.507,0.503,0.507,0.610,0.504,0.526
Random forest,0.512,0.515,0.506,0.616,0.511,0.532
AdaBoost,0.511,0.517,0.499,0.619,0.512,0.526
Neural network,0.514,0.514,0.496,0.619,0.509,0.539


## Main result: better than A?

Each model with set B, C or D against the same model with set A, on the same days (all markets and years together). McNemar's test counts days where only one of the two was right. Holm adjusts over the 21 close-to-close comparisons. The AUC range is from whole weeks drawn at random.

In [3]:
def against_a(target, s, model):
    a, b = together(target, "A", model), together(target, s, model)
    assert a.index.equals(b.index) and (a["market"] == b["market"]).all()  # the same days
    only_b, only_a, p = mcnemar(b["up"], b["p_up"] > 0.5, a["p_up"] > 0.5)
    daily = pd.DataFrame({"up": b["up"], "a": a["p_up"], "b": b["p_up"]})
    auc_gain = lambda f: roc_auc_score(f["up"], f["b"]) - roc_auc_score(f["up"], f["a"])
    low, high = bootstrap_range(daily, auc_gain)
    return {"accuracy gain, points": 100 * (right(b).mean() - right(a).mean()), "AUC gain": auc_gain(daily),
            "AUC gain, low": low, "AUC gain, high": high,
            "days only it was right": only_b, "days only A was right": only_a, "p": p}

main = pd.DataFrame({(s, model): against_a("close to close", s, model) for s in "BCD" for model in MODELS}).T
main["p (Holm)"] = holm(main["p"])
main.round(3)

accuracy gain, points  AUC gain  AUC gain, low  \
B LDA                                 -0.271    -0.003         -0.011   
  Logistic regression                  0.049    -0.001         -0.007   
  KNN                                 -0.515    -0.001         -0.010   
  SVM                                  0.049    -0.002         -0.010   
  Random forest                       -0.585     0.000         -0.007   
  AdaBoost                             0.494     0.004         -0.005   
  Neural network                      -0.320    -0.001         -0.012   
C LDA                                  0.202    -0.008         -0.019   
  Logistic regression                 -0.063    -0.009         -0.019   
  KNN                                 -0.654    -0.007         -0.019   
  SVM                                 -0.383    -0.008         -0.018   
  Random forest                       -0.891    -0.008         -0.019   
  AdaBoost                            -0.919    -0.010         -0.023   
  Neural network                      -1.100    -0.016         -0.027   
D LDA                                  7.307     0.108          0.094   
  Logistic regression                  6.924     0.106          0.092   
  KNN                                  5.999     0.092          0.079   
  SVM                                  6.486     0.101          0.086   
  Random forest                        6.541     0.102          0.088   
  AdaBoost                             7.836     0.106          0.090   
  Neural network                       7.182     0.106          0.092   

                       AUC gain, high  days only it was right  \
B LDA                           0.005                  1700.0   
  Logistic regression           0.006                   830.0   
  KNN                           0.007                  1519.0   
  SVM                           0.006                  1338.0   
  Random forest                 0.008                  1224.0   
  AdaBoost                      0.013                  2531.0   
  Neural network                0.010                  2483.0   
C LDA                           0.004                  2487.0   
  Logistic regression          -0.000                  1150.0   
  KNN                           0.004                  2351.0   
  SVM                           0.001                  1543.0   
  Random forest                 0.004                  2273.0   
  AdaBoost                      0.003                  3559.0   
  Neural network               -0.005                  2674.0   
D LDA                           0.121                  2845.0   
  Logistic regression           0.120                  2719.0   
  KNN                           0.105                  2636.0   
  SVM                           0.115                  2990.0   
  Random forest                 0.115                  3004.0   
  AdaBoost                      0.120                  3751.0   
  Neural network                0.120                  3410.0   

                       days only A was right      p  p (Holm)  
B LDA                                 1739.0  0.517     1.000  
  Logistic regression                  823.0  0.883     1.000  
  KNN                                 1593.0  0.191     1.000  
  SVM                                 1331.0  0.908     1.000  
  Random forest                       1308.0  0.099     1.000  
  AdaBoost                            2460.0  0.322     1.000  
  Neural network                      2529.0  0.525     1.000  
C LDA                                 2458.0  0.691     1.000  
  Logistic regression                 1159.0  0.868     1.000  
  KNN                                 2445.0  0.179     1.000  
  SVM                                 1598.0  0.335     1.000  
  Random forest                       2401.0  0.063     0.822  
  AdaBoost                            3691.0  0.124     1.000  
  Neural network                      2832.0  0.034     0.481  
D LDA                    

Open to close (buy at the open, sell at the close): set D against A, Holm over these 7.

In [4]:
tradeable = pd.DataFrame({model: against_a("open to close", "D", model) for model in MODELS}).T
tradeable["p (Holm)"] = holm(tradeable["p"])
tradeable.round(3)

,"accuracy gain, points",AUC gain,"AUC gain, low","AUC gain, high",days only it was right,days only A was right,p,p (Holm)
LDA,1.522,0.021,0.014,0.028,1515.0,1301.0,0.000,0.000
Logistic regression,1.259,0.017,0.011,0.023,1298.0,1121.0,0.000,0.001
KNN,0.733,0.017,0.008,0.025,1867.0,1764.0,0.090,0.090
SVM,1.579,0.019,0.011,0.028,1675.0,1453.0,0.000,0.000
Random forest,1.587,0.022,0.014,0.030,1709.0,1486.0,0.000,0.000
AdaBoost,1.579,0.016,0.005,0.026,2277.0,2055.0,0.001,0.002
Neural network,1.871,0.027,0.017,0.036,2534.0,2271.0,0.000,0.001


## Any edge at all?

Pooled accuracy minus the fair bar, per model and market. Holm adjusts within each set (42 tests), as in step 8.

In [5]:
edge = pd.DataFrame([{"target": target, "set": s, "model": model, "market": m,
                      "gain": pooled(preds[target, s, model, m])["accuracy %"] - pooled(preds[target, s, model, m])["fair bar %"],
                      "p": pooled(preds[target, s, model, m])["p vs fair"]}
                     for target, sets in RUNS.items() for s in sets for model in MODELS for m in markets])
edge["p (Holm)"] = edge.groupby(["target", "set"])["p"].transform(holm)
edge.groupby(["target", "set"]).agg(
    tests=("p", "size"), mean_gain=("gain", "mean"), above=("gain", lambda g: (g > 0).sum()),
    significant=("p", lambda p: (p < 0.05).sum()), significant_after_holm=("p (Holm)", lambda p: (p < 0.05).sum())).round(2)

tests  mean_gain  above  significant  \
target         set                                         
close to close A       42       0.07     25            1   
               B       42      -0.08     21            1   
               C       42      -0.47     12            2   
               D       42       6.95     42           42   
open to close  A       42       0.25     22            9   
               D       42       1.69     29           19   

                    significant_after_holm  
target         set                          
close to close A                         0  
               B                         0  
               C                         0  
               D                        42  
open to close  A                         3  
               D                         8

Each set's gains over the fair bar, in points (model × market).

In [6]:
for (target, s), table in edge.groupby(["target", "set"]):
    print(f"set {s}, {target}")
    display(table.pivot(index="model", columns="market", values="gain").round(1))

set A, close to close


market,India,Indonesia,Malaysia,Philippines,Singapore,Thailand
model,,,,,,
AdaBoost,-0.4,-2.9,2.5,0.2,-1.9,-0.6
KNN,0.2,-0.7,1.5,-0.5,1.3,0.5
LDA,-1.3,-0.6,0.8,0.5,0.4,-1.2
Logistic regression,0.0,-0.3,0.5,0.9,-0.3,-0.3
Neural network,0.0,-1.4,1.4,0.2,-0.7,0.1
Random forest,0.2,-1.1,1.6,1.6,1.3,0.3
SVM,0.4,0.0,0.1,0.3,-0.5,0.6


set B, close to close


market,India,Indonesia,Malaysia,Philippines,Singapore,Thailand
model,,,,,,
AdaBoost,-1.1,-1.8,1.6,1.6,-0.9,0.6
KNN,-1.4,-0.4,0.5,-0.4,-0.1,1.2
LDA,-0.4,-1.9,0.1,1.8,-1.1,-1.5
Logistic regression,0.0,-0.1,0.3,1.1,-0.7,0.2
Neural network,-1.5,-1.6,-0.7,0.6,0.3,0.7
Random forest,-0.7,-1.0,0.2,1.1,0.2,0.8
SVM,0.1,-0.2,0.2,1.7,-0.9,0.2


set C, close to close


market,India,Indonesia,Malaysia,Philippines,Singapore,Thailand
model,,,,,,
AdaBoost,-2.2,-3.5,-0.1,-1.6,1.2,-2.4
KNN,-0.9,-0.2,1.7,0.2,-1.3,-0.8
LDA,0.1,-1.0,0.7,1.0,-0.4,-0.5
Logistic regression,-0.9,0.3,-0.3,0.2,0.7,0.1
Neural network,-2.2,-2.6,-0.0,-0.4,-0.7,-1.0
Random forest,-0.8,-0.9,1.3,-0.4,-0.2,-0.3
SVM,-1.2,-0.3,1.2,-0.5,-0.4,-0.4


... (2 outputs left out)

set A, open to close


market,India,Indonesia,Malaysia,Philippines,Singapore,Thailand
model,,,,,,
AdaBoost,-0.3,3.9,1.3,0.6,-0.3,-3.6
KNN,0.6,3.8,0.8,-1.5,1.4,-1.3
LDA,-0.3,2.8,1.1,-0.3,0.5,-2.7
Logistic regression,-0.2,2.1,0.8,-0.9,0.2,-1.6
Neural network,0.5,3.1,1.6,-0.1,-1.2,-2.1
Random forest,0.8,3.5,0.9,-0.2,1.1,-3.4
SVM,-0.5,2.0,-0.7,-2.0,1.2,-0.9


set D, open to close


market,India,Indonesia,Malaysia,Philippines,Singapore,Thailand
model,,,,,,
AdaBoost,-2.6,4.6,4.9,4.6,1.1,-1.6
KNN,-0.6,2.4,2.8,4.1,-0.7,-0.0
LDA,0.4,3.1,2.6,3.7,1.4,-1.1
Logistic regression,0.7,2.3,1.8,3.8,0.1,-1.0
Neural network,-0.3,5.3,3.6,6.3,-0.2,-1.5
Random forest,-0.1,4.2,4.3,4.9,-0.0,-1.0
SVM,0.1,1.3,2.8,3.8,0.3,0.0


## 2017–2022 against 2023–2026

Accuracy gain over A in each period, in points, averaged over the seven models.

In [7]:
periods = {"2017-2022": range(2017, 2023), "2023-2026": range(2023, 2027)}
rows = {}
for target, sets in RUNS.items():
    for s in sets:
        if s == "A":
            continue
        for period, years in periods.items():
            gains = []
            for model in MODELS:
                a, b = together(target, "A", model), together(target, s, model)
                keep = a["year"].isin(years)
                gains.append(100 * (right(b)[keep].mean() - right(a)[keep].mean()))
            rows[f"{s}, {target}", period] = np.mean(gains)
pd.Series(rows).unstack().round(2)

,2017-2022,2023-2026
"B, close to close",-0.49,0.37
"C, close to close",-0.60,-0.45
"D, close to close",7.93,5.27
"D, open to close",1.65,1.13


## What tsfresh kept

For each market, the 11 training cutoffs (2015–2025): how often no feature passed (so the top 10 were used), and how many passed otherwise.

In [8]:
rows, kept = {}, Counter()
for m in markets:
    chosen = pd.read_pickle(SAVED / f"tsfresh_selection_close_to_close_{m}.pkl")
    passed = [len(cols) for cols, fell_back in chosen.values() if not fell_back]
    rows[m] = {"cutoffs": len(chosen), "fell back to the top 10": sum(f for _, f in chosen.values()),
               "features passing, median": np.median(passed) if passed else np.nan,
               "features passing, most": max(passed) if passed else np.nan}
    kept.update(c for cols, _ in chosen.values() for c in cols)
display(pd.DataFrame(rows).T)
pd.Series(dict(kept.most_common(10)), name="times kept").to_frame()

,cutoffs,fell back to the top 10,"features passing, median","features passing, most"
Indonesia,11.0,11.0,NaN,NaN
Malaysia,11.0,11.0,NaN,NaN
Philippines,11.0,10.0,1.0,1.0
Singapore,11.0,11.0,NaN,NaN
Thailand,11.0,11.0,NaN,NaN
India,11.0,11.0,NaN,NaN


,times kept
"ret__agg_linear_trend__attr_""rvalue""__chunk_len_5__f_agg_""mean""",16
"ret__cwt_coefficients__coeff_14__w_2__widths_(2, 5, 10, 20)",15
"ret__agg_linear_trend__attr_""slope""__chunk_len_5__f_agg_""mean""",15
"ret__agg_linear_trend__attr_""slope""__chunk_len_10__f_agg_""mean""",15
"ret__fft_coefficient__attr_""angle""__coeff_5",13
"ret__agg_linear_trend__attr_""rvalue""__chunk_len_5__f_agg_""max""",12
"ret__agg_linear_trend__attr_""rvalue""__chunk_len_10__f_agg_""var""",11
"ret__change_quantiles__f_agg_""mean""__isabs_False__qh_1.0__ql_0.0",11
ret__mean_change,11
ret__index_mass_quantile__q_0.6,10


## Added after looking: is set D's gain a timing leak?

Set D's gain was bigger than step 5's. If a move were dated a day late, it would line up with day t's own trading **after the open**. A move from after Asia's close should instead line up with the **overnight gap** (close t−1 → open t), and with nothing after the open.

In [9]:
from data import load_other_moves, load_us
from features import make_other_markets

us, others = load_us(include_holdout=True), load_other_moves()
rows = {}
for m, df in markets.items():
    X = make_other_markets(df, us, others, m)
    real_open = df["Open"] != df["Close"].shift(1)
    gap = np.log(df["Open"] / df["Close"].shift(1)).where(real_open)
    after_open = np.log(df["Close"] / df["Open"]).where(real_open)
    for move in ["us_ret", "vix_move", "fx_move", "oil_move"]:
        rows[move, m] = {"with the overnight gap": X[move].corr(gap), "with day t after the open": X[move].corr(after_open),
                         "with day t+1 (should be about 0)": X[move].corr(df["ret"].shift(-1))}
pd.DataFrame(rows).T.round(2)

,,with the overnight gap,with day t after the open,with day t+1 (should be about 0)
us_ret,Indonesia,0.43,0.06,-0.04
vix_move,Indonesia,-0.39,-0.05,0.02
fx_move,Indonesia,-0.02,-0.05,-0.05
oil_move,Indonesia,0.11,0.00,-0.02
us_ret,Malaysia,0.53,0.07,0.01
vix_move,Malaysia,-0.41,-0.06,-0.05
fx_move,Malaysia,-0.09,-0.07,-0.02
oil_move,Malaysia,0.21,0.05,0.01
us_ret,Philippines,0.46,0.11,-0.03
vix_move,Philippines,-0.31,-0.13,-0.02


## Added after looking: which of set D's moves carries the gain?

Logistic regression only, close to close: A plus one move at a time.

In [10]:
from classifiers import CLASSIFIERS, fit_classifier
from models import walk_forward
from more_inputs import TEST_YEARS, all_inputs

inputs = all_inputs(markets)
accuracy = lambda p: 100 * right(p).mean()
rows = {}
for m, df in markets.items():
    A, D = inputs["A"][m], inputs["D"][m]
    run = lambda X: walk_forward(df, fit_classifier, CLASSIFIERS["Logistic regression"], make_x=lambda _: X.reindex(df.index), test_years=TEST_YEARS)
    rows[m] = {"A": accuracy(run(A)), **{f"A + {move}": accuracy(run(A.join(D[[move]]))) for move in ["us_ret", "vix_move", "fx_move", "oil_move"]},
               "all of D": accuracy(preds["close to close", "D", "Logistic regression", m])}
by_move = pd.DataFrame(rows).T
display(by_move.round(1))
by_move.sub(by_move["A"], axis=0).mean().round(1).to_frame("gain over A, points (mean of markets)")

,A,A + us_ret,A + vix_move,A + fx_move,A + oil_move,all of D
Indonesia,52.5,57.2,56.8,53.4,52.8,57.7
Malaysia,50.2,60.2,57.8,49.4,50.6,58.3
Philippines,52.1,60.1,57.5,52.2,51.6,59.6
Singapore,50.7,61.1,59.0,50.9,52.2,60.6
Thailand,51.0,55.9,54.9,50.5,51.7,56.1
India,53.9,59.2,58.6,54.0,54.0,59.5


,"gain over A, points (mean of markets)"
A,0.0
A + us_ret,7.2
A + vix_move,5.7
A + fx_move,-0.0
A + oil_move,0.4
all of D,6.9


## Conclusions

**More indicators don't help. Other markets' moves do, but only because they happen after Asia's close.**

| Expectation from the plan | What happened |
|---|---|
| B: no difference from A | **As expected.** Accuracy −0.6 to +0.5 points against A, AUC −0.003 to +0.004. No comparison significant (every Holm p = 1.0) |
| C: no difference or slightly worse; most folds fall back to the top 10 | **As expected.** AUC lower for all 7 models (−0.016 to −0.007), accuracy −1.1 to +0.2 points. None significant after Holm (smallest 0.48). No tsfresh feature passed selection in 65 of the 66 training cutoffs, so the models got the 10 "least bad" features of noise |
| D, close to close: a gain in most markets, like step 5 | **As expected, in all six markets.** +6.0 to +7.8 points over A, AUC 0.61–0.62 against 0.51 (every p < 0.001). All 42 model-market pairs beat the fair bar after Holm |
| D, open to close: no difference in most markets | **Half right.** +0.7 to +1.9 points over A, significant for 6 of 7 models after Holm. But the gain is all in Indonesia, Malaysia and the Philippines (+1.3 to +6.3 over the fair bar). India, Singapore and Thailand show none |
| The same pattern in 2017–2022 and 2023–2026 | **Yes.** D's close-to-close gain is smaller in the later years (+7.9, then +5.3 points) |

**Set D's gain is step 5's US move again (added after looking).**
- **Not a timing leak.** None of the four moves lines up with day t after the open (every correlation is 0.13 or smaller in size). The US and VIX moves line up with the overnight gap (0.43 to 0.61, and −0.31 to −0.51): news from after Asia's close that the next open already contains.
- **The US move alone gives the whole gain:** +7.2 points on average for logistic regression, against +6.9 for all of D. VIX alone gives +5.7, but adds nothing on top of the US move. Currencies (0.0) and oil (+0.4) add nothing.
- Logistic regression with A plus the US move scores 55.9–61.1% here, against step 5's 57.8–61.4% with the US move alone.

**Open to close:** the markets with a gain include the Philippines and Malaysia, the two exceptions in step 5b. There the edge came from the index's opening number being partly stale, and it was smaller than trading costs. Indonesia is new here. Set A alone also beats the fair bar there for two models (+3.8 and +3.9 points, open to close), so something about Indonesia's opening numbers may be at work. I haven't looked into it. Thailand goes the other way: AdaBoost with set A is 3.6 points *below* its fair bar, significant after Holm.

**In short:** 32 standard indicators and 777 automatic features add nothing to Kim's 12: the market's own past prices hold no usable signal however they are rearranged. The only input that helps is news from the US session, and it is in the price by the next open.